In [1]:
# The following is necessary if you want to use the fast tokenizer for deberta v2 or v3
# This must be done before importing transformers
import shutil
from pathlib import Path

transformers_path = Path("/home/sthueva/anaconda3/envs/kaggle/lib/python3.7/site-packages/transformers")

input_dir = Path("../../input/nbroad/deberta-v2-3-fast-tokenizer")

convert_file = input_dir / "convert_slow_tokenizer.py"
conversion_path = transformers_path/convert_file.name

if conversion_path.exists():
    conversion_path.unlink()

shutil.copy(convert_file, transformers_path)
deberta_v2_path = transformers_path / "models" / "deberta_v2"

for filename in ['tokenization_deberta_v2.py', 'tokenization_deberta_v2_fast.py']:
    filepath = deberta_v2_path/filename
    if filepath.exists():
        filepath.unlink()

    shutil.copy(input_dir/filename, filepath)

In [2]:
#!/usr/bin/env python
# coding: utf-8

import os
import gc
import re
import json
import time
import torch
import atexit
import numpy as np
import pandas as pd
import mlcrate as mlc
from tqdm import tqdm
# from omegaconf import OmegaConf
from argparse import ArgumentParser
from sklearn.model_selection import StratifiedGroupKFold
from sklearn import metrics

import torch
import torch.nn as nn
from torch import tensor, long
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence 

from transformers import AutoTokenizer, AutoConfig, AutoModel

os.environ['CUDA_LAUNCH_BLOCKING'] = '1' 
os.environ['TOKENIZERS_PARALLELISM']= 'false'

import warnings
warnings.filterwarnings("ignore")

from data_utils import *
from dataset import FeedbackDataset


In [3]:
class Common:
    num_workers=6
    batch_size=16
    max_length=512
    num_labels=6
    label_cols=["cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions"]
    if_collate=True
    
class CFG1(Common):
    
    model="microsoft/deberta-v3-base"
    model_type='m3'
    seed=42
    n_folds=4
    trn_fold=[0, 1, 2, 3]
    loss='bce'
    weights_path='../../output/feedback-prize-english-language-learning/d3bp-f4-s42-e43-b16-d0-2e5-1e3-wd003-wm0-m3-x512'
    model_config = {'output_hidden_states': True,
                   'attention_probs_dropout_prob': 0,
                   'attention_dropout' : 0,
                   'hidden_dropout_prob': 0,
                   'hidden_dropout' : 0,
                   'layer_norm_eps': 1e-7,
                   'add_pooling_layer': False,
                  }

In [4]:
def get_text(path):
    with open(path) as f:
        content = f.read()
        f.close()
        return content

In [5]:
data_path = '../../input/feedback-prize-effectiveness/train'
essay_df =[]
for file in os.listdir(data_path):
#     print(file)
    path = os.path.join(data_path, file)
    essay_id = file.split('.')[0]
    essay = get_text(path)
    row = {"essay_id":essay_id, "essay":essay}
    essay_df.append(row)
essay_df = pd.DataFrame(essay_df)   
print(essay_df.shape)
essay_df.head()

(4191, 2)


,essay_id,essay
0,A4ADCC04C319,On a hot summer day I remembered that I have t...
1,BE604A32144C,I think that yes if there was driverless cars ...
2,8D509BBA7DD5,Touchdown! I just won the biggest game of the ...
3,20D0120E0F48,Dear TEACHER_NAME\n\nI've heard about the situ...
4,604B7BE00CDB,I think we should keep the electoral vote. My ...


In [6]:
data_path = '../../input/feedback-prize-2021/train'
essay_df1 =[]
for file in os.listdir(data_path):
#     print(file)
    path = os.path.join(data_path, file)
    essay_id = file.split('.')[0]
    essay = get_text(path)
    row = {"essay_id":essay_id, "essay":essay}
    essay_df1.append(row)
essay_df1 = pd.DataFrame(essay_df1)   
print(essay_df1.shape)
essay_df1.head()

(15594, 2)


,essay_id,essay
0,B79712C5818C,Students that don't have a grade B average or ...
1,EAE8CE22F792,Driverless cars are indeed a very complex subj...
2,A4ADCC04C319,On a hot summer day I remembered that I have t...
3,B33757C833AA,Have you ever needed advice? Most people have ...
4,466103CE7D5D,Extracurricular Activities\n\nIt's is a great ...


In [7]:
# df = pd.concat([essay_df, essay_df1], axis=0)
df = essay_df1

print(df.shape)
df.head()

(15594, 2)


,essay_id,essay
0,B79712C5818C,Students that don't have a grade B average or ...
1,EAE8CE22F792,Driverless cars are indeed a very complex subj...
2,A4ADCC04C319,On a hot summer day I remembered that I have t...
3,B33757C833AA,Have you ever needed advice? Most people have ...
4,466103CE7D5D,Extracurricular Activities\n\nIt's is a great ...


In [8]:
df.drop_duplicates(subset=['essay_id'], inplace=True)
print(df.shape)
df.head()

(15594, 2)


,essay_id,essay
0,B79712C5818C,Students that don't have a grade B average or ...
1,EAE8CE22F792,Driverless cars are indeed a very complex subj...
2,A4ADCC04C319,On a hot summer day I remembered that I have t...
3,B33757C833AA,Have you ever needed advice? Most people have ...
4,466103CE7D5D,Extracurricular Activities\n\nIt's is a great ...


In [9]:
df['essay'] = df['essay'].apply(lambda x : resolve_encodings_and_normalize(x))
df['essay'] = df['essay'].apply(lambda x : preprocessing(x))
df.head()

,essay_id,essay
0,B79712C5818C,Students that don't have a grade B average or ...
1,EAE8CE22F792,Driverless cars are indeed a very complex subj...
2,A4ADCC04C319,On a hot summer day I remembered that I have t...
3,B33757C833AA,Have you ever needed advice? Most people have ...
4,466103CE7D5D,Extracurricular Activities [BR] It's is a grea...


In [10]:
def custom_collate(data): 
    # inputs
    inputs={}
    for _key, _value in data[0]['inputs'].items():
        inputs_list = []
        for item in data:
            inputs_list.append(item['inputs'][_key])
        inputs[_key] = pad_sequence(inputs_list, batch_first=True)

    return {'inputs':inputs, 
            } 

In [11]:
# ====================================================
# Dataset
# ====================================================
class FeedbackDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, if_collate):
        self.len = len(df)
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.if_collate = if_collate # for validation
        if self.if_collate: 
            self.padding = 'do_not_pad'
        else:
            self.padding = 'max_length' # for validation
        
    def __len__(self):
        return self.len

    def __getitem__(self, index):
        text = self.df.essay[index]
#         print(text)
        
        # CREATE INPUT IDS
        encoding_inputs = self.tokenizer(text,
                                        None,
                                        add_special_tokens=True,
                                        truncation = True,
                                        max_length=self.max_length, 
                                        padding=self.padding, 
                                        )
                    
        # CONVERT TO TORCH TENSORS
        inputs = {key: tensor(val, dtype=long) for key, val in encoding_inputs.items()}

        
        return {'inputs':inputs, 
                }


In [12]:
# test_dataset = FeedbackDataset(df, 
#                                     CFG1.tokenizer, 
#                                     CFG1.max_length, 
#                                     if_collate=True,
#                                     )
# test_dataset.__len__()
# test_dataset.__getitem__(0)

In [13]:
class AttentionPool(nn.Module):
    def __init__(self, in_dim):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.LayerNorm(in_dim),
            nn.GELU(),
            nn.Linear(in_dim, 1),
        )

    def forward(self, x, mask):
        w = self.attention(x).float() #
        w[mask[:, 0]==0]=float('-inf')
        w = torch.softmax(w,1)
        x = torch.sum(w * x, dim=1)
        return x
    
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()
        
    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings
    

In [14]:
# Model3 - process all encoder layer all tokens
class Model3(nn.Module):
    def __init__(self, exp_config, pretrained=True):
        super().__init__()

        self.exp_config = exp_config
        self.config = AutoConfig.from_pretrained(self.exp_config.model)

        if self.exp_config.model_config != None:
            self.config.update(self.exp_config.model_config)

        if pretrained:
            self.model = AutoModel.from_pretrained(self.exp_config.model, config=self.config)
        else:
            self.model = AutoModel(config=self.config)
        
        self.dropout = nn.Dropout(self.config.hidden_dropout_prob)

        self.dropout1 = nn.Dropout(0.2)
        self.dropout2 = nn.Dropout(0.3)
        self.dropout3 = nn.Dropout(0.4)
        self.dropout4 = nn.Dropout(0.5)
        self.dropout5 = nn.Dropout(0.6)

        n_weights = self.config.num_hidden_layers + 1
        weights_init = torch.zeros(n_weights).float()
        weights_init.data[:-1] = -3

        self.layer_weights = torch.nn.Parameter(weights_init)
#         self.pool = AttentionPool(self.config.hidden_size)
        self.pool = MeanPooling()
        self.classifier = nn.Linear(self.config.hidden_size, self.exp_config.num_labels)

        self._init_weights(self.layer_weights)
        self._init_weights(self.pool)
        self._init_weights(self.classifier)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)


    def feature(self, inputs):
        outputs = self.model(**inputs)

        # all hidden states - all tokens
        if 'deberta' in self.exp_config.model:
            hidden_layers = outputs[1]
        else:
            hidden_layers = outputs[2]
        n_hidden_states = torch.stack([self.dropout(layer[:, :, :]) for layer in hidden_layers], dim=-1)
        hidden_states = (torch.softmax(self.layer_weights, dim=0) * n_hidden_states).sum(-1)

        return hidden_states

    def forward(self, inputs, labels=None):
        feature = self.feature(inputs)

        out = self.pool(feature, inputs['attention_mask'])

        logits1 = self.classifier(self.dropout1(out))
        logits2 = self.classifier(self.dropout2(out))
        logits3 = self.classifier(self.dropout3(out))
        logits4 = self.classifier(self.dropout4(out))
        logits5 = self.classifier(self.dropout5(out))

        logits = (logits1 + logits2 + logits3 + logits4 + logits5) / 5
        
        return logits

In [15]:
# ====================================================
# inference
# ====================================================
def inference_fn(test_loader, model, device, config):
    preds = []
    model.eval()
    model.to(device)
    tk0 = tqdm(test_loader, total=len(test_loader))
    for d in tk0:
        inputs = d["inputs"]
        #load input data to device
        for k, v in inputs.items():
            inputs[k] = v.to(device)
        with torch.no_grad():
            y_preds = model(inputs)
            y_preds = 5.0 * torch.sigmoid(y_preds).detach().to('cpu')
#             print("y_preds.shape", y_preds.shape)
            preds.append(y_preds)

    predictions = torch.cat(preds, dim=0)
    predictions = predictions.unsqueeze(0)
#     print("predictions.shape", predictions.shape)
    return predictions

In [16]:
def get_predictions(CFG, df):
    print(f"================================{CFG.model}=======================================")

    device = torch.device("cuda")
    
    if "deberta-v2" in CFG.model or "deberta-v3" in CFG.model:
        print("deberta-v2/v3")
        from transformers.models.deberta_v2.tokenization_deberta_v2_fast import DebertaV2TokenizerFast
        CFG.tokenizer = DebertaV2TokenizerFast.from_pretrained(CFG.model)
    else:
        CFG.tokenizer = AutoTokenizer.from_pretrained(CFG.model)
    
    print(f"Tokenizer all special tokens : {len(CFG.tokenizer.all_special_tokens)}")
#     CFG.tokenizer.add_special_tokens({'additional_special_tokens': ['[br]']})
#     print(f"Tokenizer all special tokens after addition : {len(CFG.tokenizer.all_special_tokens)}")
    
    df['essay_len'] = df['essay'].apply(lambda x: len(CFG.tokenizer(x)['input_ids']))
    df.sort_values(by=['essay_len'], inplace=True)
    
    test_dataset = FeedbackDataset(df, 
                                    CFG.tokenizer, 
                                    CFG.max_length, 
                                    if_collate=True,
                                    )

    if CFG.if_collate:
        test_loader = DataLoader(test_dataset,
                                 collate_fn=custom_collate, 
                                 batch_size=CFG.batch_size,
                                 shuffle=False,
                                 num_workers=CFG.num_workers, 
                                 pin_memory=True, 
                                 drop_last=False
                                )
    else:
        test_loader = DataLoader(test_dataset,
                                 batch_size=CFG.batch_size,
                                 shuffle=False,
                                 num_workers=CFG.num_workers, 
                                 pin_memory=True, 
                                 drop_last=False
                                )
    
    # MODEL 
    if CFG.model_type == 'm1':
        model = Model1(CFG, pretrained=True)
    elif CFG.model_type == 'm2':
        model = Model2(CFG, pretrained=True)
    elif CFG.model_type == 'm3':
        model = Model3(CFG, pretrained=True)
    elif CFG.model_type == 'm31':
        model = Model31(CFG, pretrained=True)
    elif CFG.model_type == 'm4':
        model = Model4(CFG, pretrained=True)
    elif CFG.model_type == 'm5':
        model = Model5(CFG, pretrained=True)
    
    model.to(device) 
#     model.model._resize_token_embeddings(len(CFG.tokenizer))
#     print(f"Token Embedding shape after adding special tokens : {model.model.get_input_embeddings()}")

    predictions = []
    for fold in CFG.trn_fold:
        print(f"------------ Fold {fold} -------------- ")
        state = torch.load(f"{CFG.weights_path}/weights/best_score_body_fold{fold}.pth", map_location=device)
       
        model.load_state_dict(state)
        prediction = inference_fn(test_loader, model, device, CFG)
        print(prediction.shape)
        
        predictions.append(prediction)
        del state, prediction; 
        gc.collect()
        torch.cuda.empty_cache()
    del model, CFG, test_dataset, test_loader
    gc.collect()
    torch.cuda.empty_cache()
    
    return predictions

In [17]:
predictions = get_predictions(CFG1, df)

================================microsoft/deberta-v3-base=======================================
deberta-v2/v3


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer all special tokens : 5


Some weights of the model checkpoint at microsoft/deberta-v3-base were not used when initializing DebertaV2Model: ['lm_predictions.lm_head.bias', 'lm_predictions.lm_head.LayerNorm.bias', 'mask_predictions.classifier.weight', 'mask_predictions.classifier.bias', 'mask_predictions.dense.bias', 'mask_predictions.dense.weight', 'mask_predictions.LayerNorm.weight', 'lm_predictions.lm_head.LayerNorm.weight', 'mask_predictions.LayerNorm.bias', 'lm_predictions.lm_head.dense.bias', 'lm_predictions.lm_head.dense.weight']
- This IS expected if you are initializing DebertaV2Model from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaV2Model from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


------------ Fold 0 -------------- 


...

torch.Size([1, 15594, 6])
------------ Fold 2 -------------- 


100%|██████████| 975/975 [07:40<00:00,  2.12it/s]


torch.Size([1, 15594, 6])
------------ Fold 3 -------------- 


100%|██████████| 975/975 [07:40<00:00,  2.12it/s]


torch.Size([1, 15594, 6])


In [22]:
predictions[0].squeeze(0).shape

torch.Size([15594, 6])

In [27]:
for fold in range(CFG1.n_folds):
    print()
    for idx, label in enumerate(CFG1.label_cols):
        print(label)
        df[label] = predictions[fold].squeeze(0)[:, idx]
    display(df.head())
    df.to_csv(f"{CFG1.weights_path}/{CFG1.weights_path.split('/')[-1]}_pseudo_fold{fold}.csv", index=False)
    #     sub


cohesion
syntax
vocabulary
phraseology
grammar
conventions


,essay_id,essay,essay_len,cohesion,syntax,vocabulary,phraseology,grammar,conventions
2925,73D6F19E24BD,The students are very mad that the principal i...,159,3.854976,3.834170,3.731154,3.863273,3.979564,3.957773
15486,470B98E980AC,What I think about phone and driving is that i...,161,3.575983,3.512046,3.710986,3.731513,3.743170,3.214009
13942,C0897AC15921,The challenge of exploring venus was being abl...,164,4.229765,4.179425,4.268696,4.245399,4.251284,4.070794
15353,4CB458757785,The authors claim to keep studing venus is a v...,165,4.327855,4.281529,4.346457,4.366097,4.397324,4.298085
7703,192A27F0608C,"Dear principal, [BR] I think that letting stud...",166,3.833515,3.819378,3.753107,3.851701,3.962617,3.983789



cohesion
syntax
vocabulary
phraseology
grammar
conventions


,essay_id,essay,essay_len,cohesion,syntax,vocabulary,phraseology,grammar,conventions
2925,73D6F19E24BD,The students are very mad that the principal i...,159,3.836878,3.814676,3.784206,3.828315,3.915658,4.026733
15486,470B98E980AC,What I think about phone and driving is that i...,161,3.485515,3.416399,3.621449,3.617856,3.507459,3.141212
13942,C0897AC15921,The challenge of exploring venus was being abl...,164,4.233634,4.196957,4.320673,4.300781,4.247533,4.193336
15353,4CB458757785,The authors claim to keep studing venus is a v...,165,4.255059,4.191922,4.321883,4.356229,4.289728,4.302876
7703,192A27F0608C,"Dear principal, [BR] I think that letting stud...",166,3.887813,3.882986,3.913766,3.936310,4.010364,4.143433



cohesion
syntax
vocabulary
phraseology
grammar
conventions


,essay_id,essay,essay_len,cohesion,syntax,vocabulary,phraseology,grammar,conventions
2925,73D6F19E24BD,The students are very mad that the principal i...,159,3.703998,3.692253,3.627785,3.685212,3.795649,3.795036
15486,470B98E980AC,What I think about phone and driving is that i...,161,3.447048,3.390100,3.615270,3.618814,3.592406,3.130210
13942,C0897AC15921,The challenge of exploring venus was being abl...,164,4.193547,4.220036,4.280560,4.244209,4.265490,4.063549
15353,4CB458757785,The authors claim to keep studing venus is a v...,165,4.269045,4.275026,4.301751,4.309546,4.364116,4.242640
7703,192A27F0608C,"Dear principal, [BR] I think that letting stud...",166,3.865578,3.851387,3.810413,3.830096,3.972402,3.988621



cohesion
syntax
vocabulary
phraseology
grammar
conventions


,essay_id,essay,essay_len,cohesion,syntax,vocabulary,phraseology,grammar,conventions
2925,73D6F19E24BD,The students are very mad that the principal i...,159,3.828027,3.869863,3.700768,3.835894,3.958580,3.958409
15486,470B98E980AC,What I think about phone and driving is that i...,161,3.453473,3.403963,3.591383,3.628004,3.567690,3.107671
13942,C0897AC15921,The challenge of exploring venus was being abl...,164,4.226395,4.236190,4.268411,4.274359,4.265786,4.116618
15353,4CB458757785,The authors claim to keep studing venus is a v...,165,4.292973,4.309148,4.337355,4.359956,4.415432,4.320799
7703,192A27F0608C,"Dear principal, [BR] I think that letting stud...",166,3.860311,3.935310,3.843389,3.879519,4.042182,4.075719
